<a href="https://colab.research.google.com/github/kutpa/Expense/blob/main/Personal_Expenses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install flask pyngrok


In [17]:
from pyngrok import ngrok

# Replace <YOUR_AUTH_TOKEN> with your actual token
ngrok.set_auth_token("2rYQrCkEAMfkeCBfa4GnRhIBUJL_576w5jUQAEgzw9A7AUvYD")


In [ ]:
import sqlite3
import os
import csv
from flask import Flask, request, redirect, url_for
from pyngrok import ngrok
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from fpdf import FPDF

###############################################################################
# Flask Initialization
###############################################################################
app = Flask(__name__)

###############################################################################
# Database Setup
###############################################################################
DB_NAME = "expenses.db"

def create_table():
    """Create the expenses table if it doesn't exist."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS expenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            item TEXT NOT NULL,
            category TEXT NOT NULL,
            price REAL NOT NULL,
            date TEXT NOT NULL,
            period TEXT NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

def get_db_connection():
    """Return a new SQLite connection."""
    return sqlite3.connect(DB_NAME)

###############################################################################
# Period Calculation (10th of current month -> 10th of next month)
###############################################################################
def get_period_range():
    today = datetime.now()
    start_date = today.replace(day=10)
    # If today's day is < 10, treat it as part of the previous period
    if today.day < 10:
        start_date = (start_date - timedelta(days=30)).replace(day=10)
    end_date = (start_date + timedelta(days=31)).replace(day=10)
    return start_date, end_date

def get_current_period():
    start_date, end_date = get_period_range()
    return f"{start_date.strftime('%d %b %Y')} - {end_date.strftime('%d %b %Y')}"

###############################################################################
# Home Route
###############################################################################
@app.route('/')
def home():
    return '''
    <h1>Welcome to the Expense Tracker</h1>
    <ul>
      <li><a href="/add">Add Expense</a></li>
      <li><a href="/view_current">View Expenses (Current Period)</a></li>
      <li><a href="/view_all">View All Expenses</a></li>
      <li><a href="/visualize">Visualize Expenses (Current Period)</a></li>
      <li><a href="/export_select">Export Expenses (Select Period)</a></li>
    </ul>
    '''

###############################################################################
# 1) Add Expense
###############################################################################
@app.route('/add', methods=['GET', 'POST'])
def add_expense():
    if request.method == 'POST':
        item = request.form.get('item')
        category = request.form.get('category')
        price = request.form.get('price')
        date_value = request.form.get('date') or datetime.now().strftime('%d/%m/%y')
        period = get_current_period()

        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO expenses (item, category, price, date, period)
            VALUES (?, ?, ?, ?, ?)
        ''', (item, category, float(price), date_value, period))
        conn.commit()
        conn.close()
        return '''
        <p>Expense Added Successfully!</p>
        <p><a href="/">Go Home</a></p>
        '''
    return '''
    <h2>Add Expense</h2>
    <form method="POST">
      <label>Item: <input type="text" name="item" required></label><br><br>
      <label>Category: <input type="text" name="category" required></label><br><br>
      <label>Price: <input type="number" step="0.01" name="price" required></label><br><br>
      <label>Date (dd/mm/yy): <input type="text" name="date"></label><br><br>
      <button type="submit">Add Expense</button>
    </form>
    <br>
    <a href="/">Go Home</a>
    '''

###############################################################################
# 2) View Expenses (Current Period)
###############################################################################
@app.route('/view_current')
def view_current():
    period = get_current_period()
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, item, category, price, date FROM expenses WHERE period = ?', (period,))
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return '''
        <h2>No expenses found for this period!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    html = f"<h2>Expenses for Period: {period}</h2><ul>"
    for (exp_id, item, category, price, date_str) in rows:
        html += (
            f"<li>ID: {exp_id} | {item} | {category} | "
            f"${price:.2f} | {date_str} | "
            f"<a href='/edit/{exp_id}'>Edit</a> | "
            f"<a href='/delete/{exp_id}'>Delete</a></li>"
        )
    html += "</ul>"
    html += '<br><a href="/">Go Home</a>'
    return html

###############################################################################
# 3) View All Expenses
###############################################################################
@app.route('/view_all')
def view_all():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, item, category, price, date, period FROM expenses')
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return '''
        <h2>No expenses found!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    html = "<h2>All Expenses</h2><ul>"
    for (exp_id, item, category, price, date_str, period_str) in rows:
        html += (
            f"<li>ID: {exp_id} | {item} | {category} | "
            f"${price:.2f} | {date_str} | {period_str} | "
            f"<a href='/edit/{exp_id}'>Edit</a> | "
            f"<a href='/delete/{exp_id}'>Delete</a></li>"
        )
    html += "</ul>"
    html += '<br><a href="/">Go Home</a>'
    return html

###############################################################################
# 4) Visualize Expenses (Current Period)
###############################################################################
@app.route('/visualize')
def visualize():
    period = get_current_period()
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT category, SUM(price) FROM expenses WHERE period = ? GROUP BY category', (period,))
    data = cursor.fetchall()
    conn.close()

    if not data:
        return '''
        <h2>No expenses found for current period to visualize!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    categories = [d[0] for d in data]
    totals = [d[1] for d in data]

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))

    # Bar Chart
    axs[0].bar(categories, totals, color='skyblue')
    axs[0].set_title(f"Expenses by Category\n({period})", fontsize=10)
    axs[0].tick_params(axis='x', rotation=45)
    axs[0].set_ylabel("Total Price")

    # Pie Chart
    axs[1].pie(totals, labels=categories, autopct="%1.1f%%", startangle=140)
    axs[1].set_title("Expense Distribution", fontsize=10)

    plt.tight_layout()
    chart_path = "current_period_charts.png"
    plt.savefig(chart_path)
    plt.close()

    html = f'''
    <h2>Visualization for Period: {period}</h2>
    <img src="/static/current_period_charts.png" width="600"><br><br>
    <a href="/">Go Home</a>
    '''
    return html

# Serving static files (the charts) from the current directory
@app.route('/static/<path:filename>')
def serve_static(filename):
    return app.send_static_file(filename)

###############################################################################
# 5) Export Expenses (Select Period) with Visuals (PDF)
###############################################################################
@app.route('/export_select', methods=['GET', 'POST'])
def export_select():
    if request.method == 'POST':
        period_selected = request.form.get('period')
        return redirect(url_for('export_result', period=period_selected))

    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT DISTINCT period FROM expenses ORDER BY period')
    periods = [row[0] for row in cursor.fetchall()]
    conn.close()

    if not periods:
        return '''
        <h2>No periods found!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    form_html = "<h2>Select a Period to Export (PDF)</h2><form method='POST'>"
    form_html += "<select name='period'>"
    for p in periods:
        form_html += f"<option value='{p}'>{p}</option>"
    form_html += "</select><br><br>"
    form_html += "<button type='submit'>Export</button></form>"
    form_html += "<br><a href='/'>Go Home</a>"
    return form_html

@app.route('/export_result/<path:period>')
def export_result(period):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('SELECT item, category, price, date FROM expenses WHERE period = ?', (period,))
    rows = cursor.fetchall()

    cursor.execute('SELECT category, SUM(price) FROM expenses WHERE period = ? GROUP BY category', (period,))
    chart_data = cursor.fetchall()
    conn.close()

    if not rows:
        return f'''
        <h2>No expenses recorded for the period {period}!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    # Generate visuals
    if chart_data:
        cats = [r[0] for r in chart_data]
        vals = [r[1] for r in chart_data]

        fig, axs = plt.subplots(1, 2, figsize=(10, 4))
        axs[0].bar(cats, vals, color='pink')
        axs[0].set_title(f"Expenses by Category\n({period})", fontsize=10)
        axs[0].tick_params(axis='x', rotation=45)
        axs[0].set_ylabel("Total Price")

        axs[1].pie(vals, labels=cats, autopct="%1.1f%%", startangle=140)
        axs[1].set_title("Expense Distribution", fontsize=10)
        plt.tight_layout()
        charts_png = "export_charts.png"
        plt.savefig(charts_png)
        plt.close()
    else:
        charts_png = None

    # Generate PDF with FPDF
    pdf_file = f"expense_report_{period.replace(' ', '_').replace('-', '_')}.pdf"
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.cell(190, 10, f"Expense Report - {period}", ln=True, align="C")

    pdf.set_font("Arial", size=12)
    pdf.ln(5)
    pdf.cell(190, 10, "Item        | Category        |  Price   |  Date", ln=True)
    pdf.cell(190, 5, "-"*60, ln=True)

    for (item, category, price, date_str) in rows:
        line_text = f"{item:<12} | {category:<15} | ${price:<8.2f} | {date_str}"
        pdf.cell(190, 8, line_text, ln=True)

    # Insert chart if exists
    if charts_png and os.path.exists(charts_png):
        pdf.add_page()
        pdf.cell(190, 10, "Expense Visuals", ln=True, align="C")
        pdf.image(charts_png, x=10, y=30, w=180)

    pdf.output(pdf_file)

    # Cleanup
    if charts_png and os.path.exists(charts_png):
        os.remove(charts_png)

    return f'''
    <h2>Exported Expenses for Period: {period}</h2>
    <p>PDF file generated: <strong>{pdf_file}</strong></p>
    <p>Find it in Colab's "Files" pane or download it.<br><br>
    <a href="/">Go Home</a></p>
    '''

###############################################################################
# 6) Edit an Entry
###############################################################################
@app.route('/edit/<int:entry_id>', methods=['GET', 'POST'])
def edit_entry(entry_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    if request.method == 'POST':
        new_item = request.form.get('item')
        new_category = request.form.get('category')
        new_price = request.form.get('price')
        new_date = request.form.get('date')

        cursor.execute('''
            UPDATE expenses
            SET item = ?, category = ?, price = ?, date = ?
            WHERE id = ?
        ''', (new_item, new_category, float(new_price), new_date, entry_id))
        conn.commit()
        conn.close()
        return '''
        <p>Entry updated successfully!</p>
        <p><a href="/">Go Home</a></p>
        '''

    # GET: Show a form with the current entry data
    cursor.execute('SELECT item, category, price, date FROM expenses WHERE id = ?', (entry_id,))
    row = cursor.fetchone()
    conn.close()

    if not row:
        return '''
        <h2>Entry not found!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    (item, category, price, date_val) = row

    return f'''
    <h2>Edit Entry (ID: {entry_id})</h2>
    <form method="POST">
      <label>Item: <input type="text" name="item" value="{item}" required></label><br><br>
      <label>Category: <input type="text" name="category" value="{category}" required></label><br><br>
      <label>Price: <input type="number" step="0.01" name="price" value="{price}" required></label><br><br>
      <label>Date (dd/mm/yy): <input type="text" name="date" value="{date_val}"></label><br><br>
      <button type="submit">Save Changes</button>
    </form>
    <br><a href="/">Go Home</a>
    '''

###############################################################################
# 7) Delete an Entry
###############################################################################
@app.route('/delete/<int:entry_id>', methods=['GET'])
def delete_entry(entry_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id FROM expenses WHERE id = ?', (entry_id,))
    row = cursor.fetchone()
    if not row:
        conn.close()
        return '''
        <h2>Entry not found!</h2>
        <p><a href="/">Go Home</a></p>
        '''

    cursor.execute('DELETE FROM expenses WHERE id = ?', (entry_id,))
    conn.commit()
    conn.close()

    return '''
    <h2>Entry deleted successfully!</h2>
    <p><a href="/">Go Home</a></p>
    '''

###############################################################################
# Main - Run Flask in Colab with Ngrok
###############################################################################
if __name__ == "__main__":
    # Create table if not exists
    create_table()

    # Start ngrok tunnel
    public_url = ngrok.connect(5000)
    print(f"Public URL: {public_url}")

    # Run Flask App
    app.run(port=5000)


Public URL: NgrokTunnel: "https://038d-35-196-158-241.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:39] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:39] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:44] "GET /add HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:47] "GET /view_current HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:49] "GET /delete/1 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jan/2025 02:15:51] "GET / HTTP/1.1" 200 -
